# 03 — Pattern Validation & Visual Tuning

**Goal:** Visually inspect detected events, assess their quality against chart intuition,
and identify which detectors need further tuning.

**Key questions:**
1. Do detected events correspond to what a human trader would identify on a chart?
2. Which detector types produce the most noise vs. genuine signals?
3. What parameter changes would improve detection quality?
4. Are forward returns after events meaningfully different from random?

**Supervisor feedback to address:**
- Technical events must be visually meaningful, not overly dense or noisy
- Detections should match what a human trader would reasonably identify
- Current parametrization likely needs further tuning

In [ ]:
import sys, os, subprocess

# --- Environment setup: works on Colab, local Jupyter, and VS Code ---
_colab = 'google.colab' in sys.modules or os.path.exists('/content')

if _colab:
    _repo_url = 'https://github.com/zeinebturki/regime-aware-ml-trading.git'
    _clone_dir = '/content/regime-aware-ml-trading'
    _project_root = os.path.join(_clone_dir, 'regime-aware-ml-trading')

    if not os.path.isdir(_project_root):
        print('Cloning repository …')
        subprocess.check_call(['git', 'clone', _repo_url, _clone_dir])

    _req = os.path.join(_clone_dir, 'regime-aware-ml-trading', 'requirements.txt')
    if os.path.isfile(_req):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', _req])
else:
    _project_root = None
    for _base in [os.path.abspath('.'), os.path.dirname(os.path.abspath(''))]:
        _cand = _base
        for _ in range(6):
            if os.path.isdir(os.path.join(_cand, 'src', 'patterns')):
                _project_root = _cand
                break
            for _sub in os.listdir(_cand):
                _nested = os.path.join(_cand, _sub)
                if os.path.isdir(_nested) and os.path.isdir(os.path.join(_nested, 'src', 'patterns')):
                    _project_root = _nested
                    break
            if _project_root:
                break
            _cand = os.path.dirname(_cand)
        if _project_root:
            break
    if _project_root is None:
        raise RuntimeError('Could not find project root (directory with src/patterns/)')

if _project_root not in sys.path:
    sys.path.insert(0, _project_root)
os.chdir(_project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_spy
from src.data.utils import compute_atr
from src.patterns.scanner import scan_all_patterns
from src.patterns.validation import EventValidator

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Imports OK')

## 1. Load data and run detectors

In [ ]:
df = load_spy()
print(f'Loaded {len(df)} bars: {df.index[0].date()} → {df.index[-1].date()}')

# Uses tuned defaults: stability filter + cooldown on S/R, R² + distinct touches on channels,
# stricter compression + cooldown on triangles, longer confirmation + cooldown on multi-top/bottom
df = scan_all_patterns(df)

event_count = df['has_event'].sum()
print(f'Events detected: {event_count} / {len(df)} ({100*event_count/len(df):.1f}%)')
print(f'Target: visually meaningful events with zero consecutive-day clustering')

## 2. Initialize the validator

In [ ]:
validator = EventValidator(df, context_bars=30, forward_horizon=10)

print('Detector types found:')
for t in validator.get_detector_types():
    count = (validator.events['event_type'] == t).sum()
    print(f'  {t:30s}  {count:>5} events')

## 3. Event density analysis

Before looking at individual events, check the overall density.
Too-dense detection = noise. Target: 10–30% of bars.

In [ ]:
# Breakdown by detector type
breakdown = validator.event_density_by_detector()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
breakdown.plot.pie(ax=axes[0], autopct='%1.1f%%', startangle=90)
axes[0].set_ylabel('')
axes[0].set_title('Event Type Breakdown')

# Bar chart
breakdown.plot.barh(ax=axes[1], color='steelblue')
axes[1].set_xlabel('Count')
axes[1].set_title('Events by Detector')

plt.tight_layout()
plt.show()

In [ ]:
# Monthly event rate
density = validator.event_density_by_month()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(density.index, density['event_rate'], width=25, color='steelblue', alpha=0.7)
ax.axhline(0.10, color='green', linestyle='--', alpha=0.5, label='10% target floor')
ax.axhline(0.30, color='red', linestyle='--', alpha=0.5, label='30% target ceiling')
ax.set_ylabel('Event Rate')
ax.set_title('Monthly Event Detection Rate')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Overall event rate: {density["event_rate"].mean():.1%}')
print(f'Months above 30%:  {(density["event_rate"] > 0.30).sum()} / {len(density)}')
print(f'Months below 10%:  {(density["event_rate"] < 0.10).sum()} / {len(density)}')

## 4. Quality report — Forward returns by detector

If a detector is meaningful, events should show directionally correct forward returns.
- Support touches → positive forward returns (bounce expected)
- Resistance touches → negative forward returns (rejection expected)
- Breakouts → strong directional moves

In [ ]:
report = validator.quality_report()
display(report.round(3))

In [ ]:
# Forward return distributions by detector type
fig, ax = plt.subplots(figsize=(12, 5))

event_types = validator.get_detector_types()
data_to_plot = []
labels = []
for etype in event_types:
    fwd = validator.events.loc[
        validator.events['event_type'] == etype, 'fwd_return'
    ].dropna() * 100
    if len(fwd) > 5:
        data_to_plot.append(fwd.values)
        labels.append(etype)

ax.boxplot(data_to_plot, labels=labels, vert=True, patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.5))
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Forward Return (%)')
ax.set_title(f'{validator.forward_horizon}-bar Forward Returns by Detector')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Visual spot-check — Support/Resistance

**What to look for:**
- Is the support/resistance level at a visually meaningful price zone?
- Does the level correspond to repeated tests (multiple bounces)?
- Or is it just a rolling min/max that doesn't represent a true level?

**Common issues:**
- Rolling-window S/R can lag and detect levels mid-trend, not at actual zones
- Proximity band too wide → flags bars that aren't really "at" the level

In [ ]:
fig = validator.plot_sample(detector='near_support', n=6, seed=42)
if fig:
    fig.suptitle('Support Touches — Visual Check', fontsize=14, y=1.02)
    plt.show()

In [ ]:
fig = validator.plot_sample(detector='near_resistance', n=6, seed=42)
if fig:
    fig.suptitle('Resistance Touches — Visual Check', fontsize=14, y=1.02)
    plt.show()

## 6. Visual spot-check — Triangles

**What to look for:**
- Is there actual convergence visible in the chart?
- Does the breakout bar clearly exit the formation?
- Are both trendlines (upper + lower) visually present?

**Common issues:**
- Regression-based convergence can find "triangles" in trendless chop
- Breakout threshold too small → noise triggers

In [ ]:
# Show all triangle types including the new upper-limit test
for tri_type in ['ascending_triangle', 'descending_triangle', 'symmetric_triangle',
                 'desc_triangle_upper_test']:
    count = (validator.events['event_type'] == tri_type).sum()
    if count > 0:
        n_show = min(6, count)
        fig = validator.plot_sample(detector=tri_type, n=n_show, seed=42)
        if fig:
            fig.suptitle(f'{tri_type} ({count} total) — Visual Check', fontsize=14, y=1.02)
            plt.show()
    else:
        print(f'No {tri_type} events detected.')

## 7. Visual spot-check — Channels

**What to look for:**
- Are two parallel trendlines clearly visible?
- Does price bounce between the lines (multiple touches)?
- Is the channel wide enough to be tradeable, but not too wide?

**Common issues:**
- Regression can fit "channels" to any trending data
- Need visual confirmation that both boundaries are respected

In [ ]:
for ch_type in ['channel_up', 'channel_down']:
    count = (validator.events['event_type'] == ch_type).sum()
    if count > 0:
        n_show = min(6, count)
        fig = validator.plot_sample(detector=ch_type, n=n_show, seed=42)
        if fig:
            fig.suptitle(f'{ch_type} ({count} total) — Visual Check', fontsize=14, y=1.02)
            plt.show()
    else:
        print(f'No {ch_type} events detected.')

## 8. Visual spot-check — Multiple Tops/Bottoms

**What to look for:**
- Are there clearly two (or more) tests of the same level?
- Do the tops/bottoms form at approximately the same price?
- Is there visible rejection after each test?

**Common issues:**
- Rolling max/min "ceiling/floor" logic can fire during trends, not just reversals
- 3-bar close slope is too short for reliable trend confirmation

In [ ]:
for mtb_type in ['multiple_top', 'multiple_bottom']:
    count = (validator.events['event_type'] == mtb_type).sum()
    if count > 0:
        n_show = min(6, count)
        fig = validator.plot_sample(detector=mtb_type, n=n_show, seed=42)
        if fig:
            fig.suptitle(f'{mtb_type} ({count} total) — Visual Check', fontsize=14, y=1.02)
            plt.show()
    else:
        print(f'No {mtb_type} events detected.')

## 9. Compare with random baseline

A good sanity check: are event forward returns statistically different
from randomly sampled bars? If not, the detector is not adding value.

In [ ]:
from scipy import stats

# Forward returns for all bars (random baseline)
all_fwd = pd.Series([
    (df['Close'].iloc[i + 10] - df['Close'].iloc[i]) / df['Close'].iloc[i]
    for i in range(len(df) - 10)
])

event_fwd = validator.events['fwd_return'].dropna()

# Two-sample t-test
t_stat, p_val = stats.ttest_ind(event_fwd, all_fwd, equal_var=False)

print(f'All bars      — mean fwd return: {all_fwd.mean()*100:+.3f}%  (n={len(all_fwd)})')
print(f'Event bars    — mean fwd return: {event_fwd.mean()*100:+.3f}%  (n={len(event_fwd)})')
print(f'T-statistic: {t_stat:.3f},  p-value: {p_val:.4f}')
print()
if p_val < 0.05:
    print('→ Event returns are statistically different from random (p < 0.05).')
else:
    print('→ No significant difference — detectors may not be adding edge.')

In [ ]:
# Per-detector comparison against random
print(f'{"Detector":<30} {"Mean %":>8} {"N":>6} {"t-stat":>8} {"p-val":>8} {"Sig?":>6}')
print('-' * 72)
for etype in validator.get_detector_types():
    fwd = validator.events.loc[
        validator.events['event_type'] == etype, 'fwd_return'
    ].dropna()
    if len(fwd) < 10:
        continue
    t, p = stats.ttest_ind(fwd, all_fwd, equal_var=False)
    sig = '  YES' if p < 0.05 else '   no'
    print(f'{etype:<30} {fwd.mean()*100:>+7.3f} {len(fwd):>6} {t:>+8.3f} {p:>8.4f} {sig}')

## 10. Tuning recommendations

Based on the visual inspection and statistics above, document which
detectors need adjustment.

In [ ]:
# Summary for tuning decisions
report = validator.quality_report()

print('=== TUNING SUMMARY ===')
print()
for _, row in report.iterrows():
    det = row['detector']
    n = int(row['count'])
    pct_pos = row['pct_positive']
    fwd_mean = row['fwd_mean_%']
    
    # Flag potential issues
    issues = []
    if n > 500:
        issues.append('HIGH COUNT — likely too many detections')
    if 45 < pct_pos < 55:
        issues.append('NEAR COIN-FLIP — no directional edge')
    if abs(fwd_mean) < 0.05:
        issues.append('NEGLIGIBLE forward return')
    
    status = ' | '.join(issues) if issues else 'OK'
    print(f'{det:<30} n={n:>5}  fwd={fwd_mean:>+.3f}%  pos={pct_pos:.1f}%  → {status}')

print()
print('Next steps:')
print('1. For detectors with HIGH COUNT: tighten parameters (wider window, stricter thresholds)')
print('2. For COIN-FLIP detectors: reconsider the detection logic or add confirmation')
print('3. For OK detectors: proceed to regime detection (notebook 04)')

## 11. Interactive single-event exploration

Use this cell to manually inspect any specific event date.

In [ ]:
# Change this date to inspect any specific event
# Pick a date from the events list:
sample_dates = validator.events.index[:20].strftime('%Y-%m-%d').tolist()
print('First 20 event dates:', sample_dates[:10], '...')

# Uncomment and set a date to inspect:
# fig, ax = validator.plot_event(pd.Timestamp('2020-03-12'))
# plt.show()

---

## Conclusion & Next Steps

**Tuning results (Iteration 3):**

| Metric | Before tuning | After tuning |
|--------|:---:|:---:|
| Total events | 1,067 (26.5%) | **183 (4.5%)** |
| Consecutive-day clusters | 202 | **0** |
| Detectors with clustering | 4/4 | **0/4** |
| New signal types | — | `desc_triangle_upper_test` |

**Parameter changes applied:**
- **S/R:** added level stability (3-bar flat) + 10-bar cooldown
- **Channels:** added R²≥0.70 + distinct touch counting (5-bar separation) + min_touches=4 + cooldown
- **Triangles:** compression 3%→5% + cooldown + new upper-limit test signal
- **Multi top/bottom:** confirmation 3→5 bars + cooldown

**Design decision:** Accepted 4.5% event rate — supervisor prioritized visually
meaningful patterns over density. 183 high-quality events with no clustering is
better than 1,067 noisy events.

**Next notebook:** `04_hmm_regimes.ipynb` — Fit HMM market regimes using
daily log returns and rolling volatility. Regimes will later be joined
back to the event dataset as context for ML models.